# Análise Comparativa de Modelos para Sumarização de Textos Científicos

Este notebook apresenta a comparação entre os modelos GPT-2, DistilGPT-2 e BERTimbau
na tarefa de sumarização automática de textos científicos em português.

In [ ]:
import sys
import os
import json
import time

sys.path.insert(0, os.path.abspath(".."))

from summarization import ModelLoader, Summarizer, TextPreprocessor, ExtractiveSummarizer
from evaluation import SummaryEvaluator, SemanticEvaluator
from data.corpus import Corpus
from utils import Timer

import matplotlib.pyplot as plt
import numpy as np

## 1. Carregamento do Corpus

Utilizamos um corpus de 10 textos científicos em português com resumos de referência.

In [ ]:
corpus = Corpus()
samples = corpus.get_builtin_samples()
texts, references = corpus.get_texts_and_references()

print(f"Total de amostras: {len(samples)}\n")
for s in samples:
    print(f"  - {s['id']}: {s['title']}")

## 2. Configuração dos Modelos

Modelos avaliados:
- **GPT-2**: modelo generativo causal pré-treinado
- **DistilGPT-2**: versão destilada do GPT-2 (mais leve)
- **BERTimbau**: modelo BERT para português (sumarização extrativa)

In [ ]:
MODELS = ["gpt2", "distilgpt2", "bertimbau"]
NUM_SAMPLES = 5

test_texts = texts[:NUM_SAMPLES]
test_references = references[:NUM_SAMPLES]

print(f"Modelos: {MODELS}")
print(f"Amostras para teste: {NUM_SAMPLES}")

## 3. Geração de Resumos

Executamos cada modelo sobre as amostras de teste e medimos o tempo de execução.

In [ ]:
results = {}

for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Processando: {model_name}")
    print(f"{'='*60}")

    summaries = []
    with Timer() as t:
        try:
            if model_name == "bertimbau":
                summarizer = ExtractiveSummarizer()
                for text in test_texts:
                    summary = summarizer.generate_summary(text, num_sentences=3)
                    summaries.append(summary)
            else:
                summarizer = Summarizer(model_name=model_name)
                for text in test_texts:
                    summary = summarizer.generate_summary(text, max_length=150)
                    summaries.append(summary)
        except Exception as e:
            print(f"Erro ao processar {model_name}: {e}")
            continue

    results[model_name] = {
        "summaries": summaries,
        "time": t.elapsed,
        "time_str": t.elapsed_str
    }
    print(f"Tempo total: {t.elapsed_str}")
    print(f"Resumos gerados: {len(summaries)}")

## 4. Exemplos de Resumos Gerados

Comparação lado a lado dos resumos produzidos por cada modelo.

In [ ]:
for i in range(min(3, NUM_SAMPLES)):
    print(f"\n{'='*70}")
    print(f"AMOSTRA {i+1}: {samples[i]['title']}")
    print(f"{'='*70}")
    print(f"\nTexto original (primeiras 200 palavras):")
    print(f"  {' '.join(test_texts[i].split()[:200])}...")
    print(f"\nResumo de referencia:")
    print(f"  {test_references[i]}")

    for model_name in MODELS:
        if model_name in results and i < len(results[model_name]["summaries"]):
            print(f"\nResumo [{model_name}]:")
            print(f"  {results[model_name]['summaries'][i]}")

## 5. Avaliação com ROUGE

Métricas ROUGE medem a sobreposição de n-gramas entre o resumo gerado e o resumo de referência:
- **ROUGE-1**: unigramas
- **ROUGE-2**: bigramas
- **ROUGE-L**: maior subsequência comum

In [ ]:
evaluator = SummaryEvaluator()
rouge_results = {}

for model_name in MODELS:
    if model_name not in results or not results[model_name]["summaries"]:
        continue

    scores = evaluator.evaluate_batch(
        results[model_name]["summaries"],
        test_references
    )
    rouge_results[model_name] = scores

    print(f"\n--- {model_name.upper()} ---")
    evaluator.format_results(scores)

## 6. Tabela Comparativa

In [ ]:
print(f"{'Modelo':<15} {'ROUGE-1 F':>12} {'ROUGE-2 F':>12} {'ROUGE-L F':>12} {'Tempo':>10}")
print("-" * 65)

for model_name in MODELS:
    if model_name not in rouge_results:
        continue
    r = rouge_results[model_name]
    print(
        f"{model_name:<15} "
        f"{r['rouge1']['fmeasure']:>12.4f} "
        f"{r['rouge2']['fmeasure']:>12.4f} "
        f"{r['rougeL']['fmeasure']:>12.4f} "
        f"{results[model_name]['time_str']:>10}"
    )

## 7. Visualização dos Resultados

In [ ]:
plt.style.use("seaborn-v0_8-darkgrid")

model_names = [m for m in MODELS if m in rouge_results]
rouge1_scores = [rouge_results[m]["rouge1"]["fmeasure"] for m in model_names]
rouge2_scores = [rouge_results[m]["rouge2"]["fmeasure"] for m in model_names]
rougeL_scores = [rouge_results[m]["rougeL"]["fmeasure"] for m in model_names]

x = np.arange(len(model_names))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width, rouge1_scores, width, label="ROUGE-1", color="#2196F3")
bars2 = ax.bar(x, rouge2_scores, width, label="ROUGE-2", color="#FF9800")
bars3 = ax.bar(x + width, rougeL_scores, width, label="ROUGE-L", color="#4CAF50")

ax.set_xlabel("Modelo")
ax.set_ylabel("F-Score")
ax.set_title("Comparacao ROUGE entre Modelos")
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()
ax.set_ylim(0, 1)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.3f}", xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("../experiments/rouge_comparison.png", dpi=150)
plt.show()

## 8. Tempo de Execução

In [ ]:
times = [results[m]["time"] for m in model_names]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#2196F3", "#FF9800", "#4CAF50"]
bars = ax.bar(model_names, times, color=colors[:len(model_names)])

ax.set_xlabel("Modelo")
ax.set_ylabel("Tempo (segundos)")
ax.set_title("Tempo de Execucao por Modelo")

for bar in bars:
    height = bar.get_height()
    ax.annotate(f"{height:.1f}s", xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha="center", va="bottom")

plt.tight_layout()
plt.savefig("../experiments/time_comparison.png", dpi=150)
plt.show()

## 9. Salvando Resultados

In [ ]:
export_data = {}
for model_name in model_names:
    export_data[model_name] = {
        "rouge_scores": rouge_results[model_name],
        "execution_time": results[model_name]["time"],
        "num_samples": len(results[model_name]["summaries"]),
        "summaries": results[model_name]["summaries"]
    }

os.makedirs("../experiments", exist_ok=True)
with open("../experiments/resultados_comparacao.json", "w", encoding="utf-8") as f:
    json.dump(export_data, f, ensure_ascii=False, indent=2)

print("Resultados salvos em experiments/resultados_comparacao.json")

## 10. Conclusões Preliminares

A análise acima permite identificar:

1. Qual modelo obteve melhores scores ROUGE (qualidade do resumo)
2. Trade-off entre qualidade e velocidade de cada modelo
3. Diferenças entre abordagem extrativa (BERTimbau) e generativa (GPT-2/DistilGPT-2)

**Próximos passos:**
- Adicionar LLaMA-2 à comparação quando acesso for aprovado
- Avaliar similaridade semântica com BERTScore
- Aumentar corpus de avaliação com mais textos científicos
- Testar diferentes parâmetros de geração (temperatura, top_p)